# Optuna hyperparameter optimization

### Import libraries and set configs

In [1]:
import ast
import json
import pandas as pd

import optuna


class CFG:
    n_trials = 250
    # maximum number of simultaneously opened trades for backtest metric
    max_num_simult_trades = 100
    # significance level, that is used to conduct a t-test between 2 models
    optimize_alpha = 0.2
    n_repeats = 1
    n_folds = 8
    min_precision = 0.5
    TP = 0.05
    SL = 0.05
    slippage = 0.002

/home/alex/Repos/sigbot/.venv/lib/python3.12/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load the train data

In [2]:
train_df = pd.read_pickle("data/train_df.pkl")

# all data for the last 90 days are test
test_date = train_df["time"].max() - pd.to_timedelta(90, unit="D")

profitable_hours_df = pd.read_csv("data/profitable_hours.csv")
latest = profitable_hours_df.iloc[-1]
buy_hours = ast.literal_eval(latest["profitable_buy_hours"])
sell_hours = ast.literal_eval(latest["profitable_sell_hours"])

buy_mask = (train_df["ttype"] == "buy") & (train_df["time"].dt.hour.isin(buy_hours))
sell_mask = (train_df["ttype"] == "sell") & (train_df["time"].dt.hour.isin(sell_hours))
train_df = train_df[buy_mask | sell_mask].reset_index(drop=True)

fi = pd.read_csv("model/features/feature_importance.csv")

### Optimize

In [ ]:
from utils.optimization_utils import make_objective

with open("model/bybit_tickers.json", "r") as f:
    bybit_tickers = json.load(f)

df_optuna_more_info = pd.DataFrame(columns=["result", "backtest_result", "oof_conf_score",
                                            "profit_objects", "oof_conf_obj_num", "scores"])
df_optuna_more_info.to_csv("model/optuna/optuna_lgbm_info.csv", index=False)

objective = make_objective(
    train_df, 
    test_date, 
    fi, 
    bybit_tickers, 
    TP=CFG.TP - CFG.slippage,
    SL=CFG.SL + CFG.slippage,
    n_folds=CFG.n_folds, 
    optimize_alpha=CFG.optimize_alpha, 
    min_precision=CFG.min_precision,
)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=CFG.n_trials)

print("Number of finished trials: {}".format(len(study.trials)))

print("Best trial:")
trial = study.best_trial

print("  Value: {}".format(trial.value))

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

df_optuna = study.trials_dataframe()
df_optuna = df_optuna.sort_values("value", ascending=False)
# df_optuna.to_csv("optuna/optuna_lgbm.csv", index=False)

display(df_optuna.head(10))

/home/alex/Repos/sigbot/.venv/lib/python3.12/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[I 2026-06-10 14:57:43,799] A new study created in memory with name: no-name-3e8109f8-6489-4ad1-9cae-91d009788b50
/home/alex/Repos/sigbot/ml/utils/optimization_utils.py:209: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_optuna_more_info = pd.concat([df_optuna_more_info, tmp])
[I 2026-06-10 14:59:01,077] Trial 0 finished with value: 29.863893858079905 and parameters: {'boosting_type': 'gbdt', 'n_esti

avg conf score 122.30138196647032 is better than best score 31.87056848084876, but p-value 0.9997575148849841 is more than alpha 0.2


[I 2026-06-10 15:24:35,123] Trial 4 finished with value: 34.2590196788057 and parameters: {'boosting_type': 'dart', 'n_estimators': 1571, 'learning_rate': 0.001672340815063068, 'reg_alpha': 0.0005037259309032895, 'reg_lambda': 0.09994319690358727, 'max_depth': 10, 'num_leaves': 148, 'colsample_bytree': 0.8703599709419501, 'max_bin': 182, 'is_unbalance': True, 'high_bound': 0.3973862435889488, 'low_bound': 0.030679051289443517, 'feature_num': 48, 'corr_thresh': 0.598481989700806, 'sample_weight': 'linear', 'subsample': 0.7439563207471149}. Best is trial 4 with value: 34.2590196788057.
[I 2026-06-10 15:29:03,252] Trial 5 finished with value: 34.25909615642656 and parameters: {'boosting_type': 'goss', 'n_estimators': 2910, 'learning_rate': 0.00019241339061608676, 'reg_alpha': 0.00041833260227176116, 'reg_lambda': 0.013968556512169448, 'max_depth': 5, 'num_leaves': 320, 'colsample_bytree': 0.5174266950482798, 'max_bin': 201, 'is_unbalance': True, 'high_bound': 0.5697552513750245, 'low_boun

avg conf score 49.65332625149383 is better than best score 34.2590196788057, but p-value 0.9999950320840699 is more than alpha 0.2


[I 2026-06-10 15:34:21,657] Trial 6 finished with value: 19.406459781162205 and parameters: {'boosting_type': 'gbdt', 'n_estimators': 1833, 'learning_rate': 0.0013875499619007254, 'reg_alpha': 0.005838629075471185, 'reg_lambda': 8.596993018912196e-08, 'max_depth': 8, 'num_leaves': 45, 'colsample_bytree': 0.685204748371134, 'max_bin': 145, 'is_unbalance': False, 'high_bound': 0.5011710406193406, 'low_bound': 0.029094595843100993, 'feature_num': 345, 'corr_thresh': 0.8464444870813689, 'sample_weight': 'linear', 'subsample': 0.5626681801723266}. Best is trial 5 with value: 34.25909615642656.
[I 2026-06-10 15:38:34,196] Trial 7 finished with value: 105.82232917554 and parameters: {'boosting_type': 'dart', 'n_estimators': 990, 'learning_rate': 0.16541529012845094, 'reg_alpha': 7.976586140154063e-05, 'reg_lambda': 5.001105260823771e-08, 'max_depth': 6, 'num_leaves': 383, 'colsample_bytree': 0.733832298024343, 'max_bin': 63, 'is_unbalance': True, 'high_bound': 0.6291622485495475, 'low_bound':

avg conf score 145.85417088821345 is better than best score 105.82232917554, but p-value 0.9999305090264714 is more than alpha 0.2


[I 2026-06-10 16:05:14,350] Trial 13 finished with value: 105.90635139531199 and parameters: {'boosting_type': 'dart', 'n_estimators': 2093, 'learning_rate': 0.014506969763513544, 'reg_alpha': 5.600369348275333e-08, 'reg_lambda': 1.604252571492189e-08, 'max_depth': 7, 'num_leaves': 361, 'colsample_bytree': 0.8108976202876785, 'max_bin': 83, 'is_unbalance': True, 'high_bound': 0.6488550601451711, 'low_bound': 0.06715812864376904, 'feature_num': 430, 'corr_thresh': 0.6782767201007662, 'sample_weight': 'cos', 'subsample': 0.6412649181656903}. Best is trial 13 with value: 105.90635139531199.


avg conf score 110.72809368996602 is better than best score 105.82511102719276, but p-value 0.9834304190516404 is more than alpha 0.2


[I 2026-06-10 16:09:56,676] Trial 14 finished with value: 164.31944979367262 and parameters: {'boosting_type': 'goss', 'n_estimators': 2231, 'learning_rate': 0.008564694335256995, 'reg_alpha': 1.0177686293542195e-08, 'reg_lambda': 4.646156666503176e-06, 'max_depth': 8, 'num_leaves': 238, 'colsample_bytree': 0.8045867564610523, 'max_bin': 99, 'is_unbalance': True, 'high_bound': 0.6489462614142889, 'low_bound': 0.06939282695524202, 'feature_num': 393, 'corr_thresh': 0.6878903121878491, 'sample_weight': 'cos'}. Best is trial 14 with value: 164.31944979367262.
[I 2026-06-10 16:15:06,218] Trial 15 finished with value: 66.35500655881066 and parameters: {'boosting_type': 'goss', 'n_estimators': 2137, 'learning_rate': 0.030353417470394845, 'reg_alpha': 1.8506926157981784e-08, 'reg_lambda': 1.1026013611571641e-08, 'max_depth': 8, 'num_leaves': 195, 'colsample_bytree': 0.898464643060131, 'max_bin': 89, 'is_unbalance': True, 'high_bound': 0.643614485803387, 'low_bound': 0.06452346015985293, 'feat